In [1]:
import pandas as pd
import numpy as np
import os
import re

In [6]:
df = pd.read_parquet('./data/train/1_회원정보_train.parquet')

In [7]:
df['Segment'].value_counts()

Segment
E    1922052
D     349242
C     127590
A        972
B        144
Name: count, dtype: int64

In [11]:
# 파일 경로 리스트
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

### C/D만 추출하여 `ID`, `기준년월`로 병합

In [12]:
# 1. Segment 정보 불러오기 (1번 파일에서만 Segment 존재)
segment_df = pd.read_parquet('./data/train/1_회원정보_train.parquet')[['ID', '기준년월', 'Segment']]
segment_df['ID'] = segment_df['ID'].astype(str)

# 2. C/D인 ID-기준년월 조합만 추출
cd_keys = segment_df[segment_df['Segment'].isin(['C', 'D'])][['ID', '기준년월']]

# 전처리 함수 (문자형 → 숫자형)
def preprocess(df):
    df_cleaned = df.copy()
    obj_cols = df_cleaned.select_dtypes(include='object').columns
    for col in obj_cols:
        sample_values = df_cleaned[col].dropna().astype(str).unique()
        if all(re.fullmatch(r'\d+대', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.replace("대", "").astype(int)
        elif all(re.fullmatch(r'\d+개', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.replace("개", "").astype(int)
        elif all(re.fullmatch(r'\d+회 이상', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.extract(r'(\d+)').astype(int)
        elif all(re.fullmatch(r'\d+일 이상', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.extract(r'(\d+)').astype(int)
        elif all(re.fullmatch(r'\d{2}\.\d+만원\+', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.extract(r'(\d{2})').astype(int)
        elif all(re.fullmatch(r'[A-Z]', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].astype('category').cat.codes
        else:
            df_cleaned[col] = df_cleaned[col].astype('category').cat.codes
    return df_cleaned.fillna(0)

In [13]:
# 5. 출력 폴더
os.makedirs('./corr_output', exist_ok=True)

# 6. 반복
for path in paths:
    name = os.path.basename(path).replace('_train.parquet', '')
    df_part = pd.read_parquet(path)
    df_part['ID'] = df_part['ID'].astype(str)

    # C/D만 필터링 (ID + 기준년월 기준)
    df_part = pd.merge(df_part, cd_keys, on=['ID', '기준년월'], how='inner')

    # Segment 병합 (1번 파일은 이미 포함)
    if name == '1_회원정보':
        df_merged = df_part.copy()
    else:
        df_merged = pd.merge(df_part, segment_df, on=['ID', '기준년월'], how='left')

    if 'Segment' not in df_merged.columns or df_merged['Segment'].isna().all():
        print(f"[경고] {name} → Segment 병합 실패 → 건너뜀")
        continue

    # Segment 인코딩 (C=0, D=1)
    df_merged['Segment'] = df_merged['Segment'].astype('category').cat.codes

    # 전처리
    df_proc = preprocess(df_merged)

    # 상관계수 계산
    try:
        corr = df_proc.corr(numeric_only=True)['Segment'].drop('Segment', errors='ignore')
        corr.to_csv(f'./corr_output/{name}_corr_C_D.csv', encoding='utf-8-sig')
        print(f"[완료] {name}: 상관계수 저장")
    except Exception as e:
        print(f"[에러] {name}: {e}")

print("\n완료: ID+기준년월 기준 병합 + C/D 상관계수 저장")


[완료] 1_회원정보: 상관계수 저장
[완료] 2_신용정보: 상관계수 저장
[완료] 3_승인매출정보: 상관계수 저장
[완료] 4_청구입금정보: 상관계수 저장
[완료] 5_잔액정보: 상관계수 저장
[완료] 6_채널정보: 상관계수 저장
[완료] 7_마케팅정보: 상관계수 저장
[완료] 8_성과정보: 상관계수 저장

완료: ID+기준년월 기준 병합 + C/D 상관계수 저장


In [ ]:
import glob

# 상관계수 결과 파일들 불러오기
corr_files = sorted(glob.glob('./corr_output/*_corr_C_D.csv'))

# 파일별로 읽고 통합
df_list = []
for file in corr_files:
    source_name = os.path.basename(file).replace('_corr_C_D.csv', '')  # 파일명만 추출
    df = pd.read_csv(file, index_col=0)
    df = df.reset_index().rename(columns={'index': 'feature', df.columns[0]: 'correlation'})
    df['source'] = source_name
    df_list.append(df)

# 전체 합치기
combined_corr = pd.concat(df_list, ignore_index=True)

# 저장
combined_corr.to_csv('./corr_output/전체_C_D_상관계수_통합.csv', index=False, encoding='utf-8-sig')

print("완료: './corr_output/전체_C_D_상관계수_통합.csv'")

완료: './corr_output/전체_C_D_상관계수_통합.csv'


In [3]:
df = pd.read_csv('./corr_output/CD/전체_C_D_상관계수_통합.csv')

In [4]:
corr_df = df[df['correlation'].abs() >= 0.3]
corr_df

,feature,correlation,source
42,이용금액_R3M_신용체크,-0.306900,1_회원정보
47,_1순위카드이용금액,-0.318514,1_회원정보
138,이용금액_일시불_B0M,-0.303103,3_승인매출정보
166,이용금액_일시불_R12M,-0.326401,3_승인매출정보
492,정상청구원금_B0M,-0.423459,3_승인매출정보
494,정상입금원금_B0M,-0.314765,3_승인매출정보
496,정상청구원금_B2M,-0.416269,3_승인매출정보
498,정상입금원금_B2M,-0.305066,3_승인매출정보
500,정상청구원금_B5M,-0.438868,3_승인매출정보
502,정상입금원금_B5M,-0.309526,3_승인매출정보


In [5]:
corr_df = df[df['correlation'].abs() >= 0.4]
corr_df

,feature,correlation,source
492,정상청구원금_B0M,-0.423459,3_승인매출정보
496,정상청구원금_B2M,-0.416269,3_승인매출정보
500,정상청구원금_B5M,-0.438868,3_승인매출정보
